# Level 2 Task 2: Time Series Analysis

This notebook analyzes AAPL stock prices over time to study trends, smoothing behavior, seasonality, and residual variation.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

In [ ]:
base_dir = Path.cwd()
if not (base_dir / "data" / "aapl_stock_prices.csv").exists():
    base_dir = Path("Level_2_Intermediate") / "Task_2_Time_Series_Analysis"

data_path = base_dir / "data" / "aapl_stock_prices.csv"
output_dir = base_dir / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid")

In [ ]:
df = pd.read_csv(data_path)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)
print("Rows:", len(df))
print("Date range:", df["date"].min().date(), "to", df["date"].max().date())
df.head()

## Moving Averages

In [ ]:
df["close_30_ma"] = df["close"].rolling(window=30, min_periods=1).mean()
df["close_90_ma"] = df["close"].rolling(window=90, min_periods=1).mean()
df[["date", "close", "close_30_ma", "close_90_ma"]].head()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(df["date"], df["close"], color="#c7c7c7", linewidth=1.2, label="Close")
plt.plot(df["date"], df["close_30_ma"], color="#2ca02c", linewidth=2.0, label="30-day MA")
plt.plot(df["date"], df["close_90_ma"], color="#d62728", linewidth=2.0, label="90-day MA")
plt.title("AAPL Closing Price with Moving Averages", fontweight="bold")
plt.xlabel("Date")
plt.ylabel("Price")
plt.legend()
plt.tight_layout()
plt.show()

## Additive-Style Decomposition

In [ ]:
df["trend"] = df["close"].rolling(window=30, center=True, min_periods=15).mean()
df["detrended"] = df["close"] - df["trend"]
seasonal_by_month = df.groupby(df["date"].dt.month)["detrended"].mean().fillna(0)
seasonal_by_month = seasonal_by_month - seasonal_by_month.mean()
df["seasonal"] = df["date"].dt.month.map(seasonal_by_month)
df["residual"] = df["close"] - df["trend"] - df["seasonal"]
df[["date", "close", "trend", "seasonal", "residual"]].head()

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(12, 12), sharex=True)
axes[0].plot(df["date"], df["close"], color="#1f77b4")
axes[0].set_title("Observed Close Price")
axes[1].plot(df["date"], df["trend"], color="#2ca02c")
axes[1].set_title("Trend")
axes[2].plot(df["date"], df["seasonal"], color="#ff7f0e")
axes[2].set_title("Seasonal Component")
axes[3].plot(df["date"], df["residual"], color="#7f7f7f")
axes[3].set_title("Residual Component")
axes[3].set_xlabel("Date")
plt.tight_layout()
plt.show()

## Monthly Seasonality

In [ ]:
df["month_name"] = df["date"].dt.month_name()
monthly_avg_close = df.groupby("month_name")["close"].mean().reindex([
    "January", "February", "March", "April", "May", "June",
    "July", "August", "September", "October", "November", "December"
]).round(3)
monthly_avg_close

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(x=monthly_avg_close.index, y=monthly_avg_close.values, color="#3498db")
plt.title("Average Monthly Closing Price", fontweight="bold")
plt.xlabel("Month")
plt.ylabel("Average Close Price")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Daily Returns

In [ ]:
df["daily_return_pct"] = df["close"].pct_change() * 100
df["daily_return_pct"].describe().round(3)

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df["daily_return_pct"].dropna(), bins=40, kde=True, color="#8e44ad")
plt.title("Distribution of Daily Percentage Returns", fontweight="bold")
plt.xlabel("Daily Return (%)")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()